In [1]:
!pip install pandas 

In [2]:
import random 
import numpy as np 
import pandas as pd 

In [3]:
INTENTS = ["attack", "retreat", "reposition", "idle"]

def get_motion_class(vx, vy):
    if abs(vx) > abs(vy):
        return 1 if vx > 0 else -1  
    else:
        return 2 if vy > 0 else -2

In [4]:
def generate_sequence(seq_len=10, intent="attack"):
    sequence = []

    X, Y = random.uniform(-100, 100), random.uniform(50, 200)

    for t in range(seq_len):

        if intent == "attack":
            vx = -X * 0.05 + np.random.randn()
            vy = -Y * 0.1 + np.random.randn()

        elif intent == "retreat":
            vx = X * 0.05 + np.random.randn()
            vy = Y * 0.1 + np.random.randn()

        elif intent == "reposition":
            if random.random() < 0.5:
                vx = 5 + np.random.randn()   
            else:
                vx = -5 + np.random.randn()  

            vy = np.random.randn() * 0.5  

        else:
            vx = np.random.randn() * 0.5
            vy = np.random.randn() * 0.5

        X += vx
        Y += vy

        speed = np.sqrt(vx**2 + vy**2)
        distance = np.sqrt(X**2 + Y**2)
        motion_class = get_motion_class(vx, vy)
        threat = np.exp(-distance / 100)

        sequence.append([
            t, X, Y,
            vx, vy, speed,
            motion_class,
            distance,
            threat
        ])

    return sequence, intent

In [5]:
def generate_dataset(n_samples, seq_len=10):
    rows = []

    for seq_id in range(n_samples):
        intent = random.choice(INTENTS)
        sequence, label = generate_sequence(seq_len, intent)

        for step in sequence:
            t, X, Y, vx, vy, speed, motion_class, distance, threat = step

            rows.append({
                "sequence_id": seq_id,
                "t": t,
                "X": X,
                "Y": Y,
                "vx": vx,
                "vy": vy,
                "speed": speed,
                "motion_class": motion_class,
                "distance": distance,
                "threat": threat,
                "intent": label
            })

    df = pd.DataFrame(rows)
    return df


In [6]:
data = generate_dataset(n_samples=5000, seq_len=10)

In [7]:
data

,sequence_id,t,X,Y,vx,vy,speed,motion_class,distance,threat,intent
0,0,0,-60.946138,196.966037,-4.469285,-0.450112,4.491894,-1,206.179658,0.127225,reposition
1,0,1,-55.693151,197.403167,5.252986,0.437131,5.271143,1,205.109087,0.128595,reposition
2,0,2,-60.620444,196.570349,-4.927292,-0.832818,4.997179,-1,205.705470,0.127830,reposition
3,0,3,-65.290447,196.784927,-4.670004,0.214578,4.674931,-1,207.333428,0.125766,reposition
4,0,4,-70.507729,197.183724,-5.217282,0.398797,5.232501,-1,209.410509,0.123180,reposition
...,...,...,...,...,...,...,...,...,...,...,...
49995,4999,5,35.331296,115.564848,0.611553,0.556272,0.826702,1,120.845085,0.298660,idle
49996,4999,6,35.642107,115.796848,0.310811,0.232000,0.387849,1,121.158037,0.297726,idle
49997,4999,7,36.173334,115.432187,0.531227,-0.364662,0.644344,1,120.967350,0.298295,idle
49998,4999,8,36.379956,115.689068,0.206622,0.256881,0.329667,2,121.274324,0.297380,idle


In [8]:
data['motion_class'].value_counts()


# 2    15924 -> retreat
# -2   14656 -> attack
#  1   9783 -> flank right
# -1   9637 -> flank left

motion_class
 2    15864
-2    14997
 1     9577
-1     9562
Name: count, dtype: int64

In [9]:
data.to_csv('intent_data.csv', index=False)